# Нейросеть

In [1]:
import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [2]:
X_train = pd.read_csv('data/X_train.csv')
X_test = pd.read_csv('data/X_test.csv')
y_train = pd.read_csv('data/y_train.csv')
y_test = pd.read_csv('data/y_test.csv')

In [3]:
X_train.head(3)

,age,education.num,capital.gain,capital.loss,hours.per.week,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,...,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Unknown,native.country_Vietnam,native.country_Yugoslavia
0,1.498786,1.520923,-0.145313,-0.215487,-0.031025,-0.261482,-0.013856,-1.520489,-0.186659,-0.288699,...,-0.058882,-0.019597,-0.049239,-0.039705,-0.02319,-0.025555,0.33978,-0.135555,-0.045579,-0.020554
1,-1.138419,-0.422173,-0.145313,-0.215487,-0.031025,-0.261482,-0.013856,0.657683,-0.186659,-0.288699,...,-0.058882,-0.019597,-0.049239,-0.039705,-0.02319,-0.025555,0.33978,-0.135555,-0.045579,-0.020554
2,1.352274,-0.033554,0.893630,-0.215487,0.374998,-0.261482,-0.013856,0.657683,-0.186659,-0.288699,...,-0.058882,-0.019597,-0.049239,-0.039705,-0.02319,-0.025555,0.33978,-0.135555,-0.045579,-0.020554


In [4]:
DEVICE = 'cpu'
BATCH_SIZE = 100

### Гипотеза 1
Простая модель с одним скрытым слоем размерности 32

In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self, in_features, hidden_size):
        super().__init__()
        self.l1 = nn.Linear(in_features, hidden_size)
        self.act = nn.ReLU()
        self.outl = nn.Linear(hidden_size, 1)
        self.final_act = nn.Sigmoid()

    def forward(self, x):
        y = self.l1(x)
        y = self.act(y)
        y = self.outl(y)
        y = self.final_act(y)
        return y

In [6]:
model = NeuralNetwork(98, 32)

In [7]:
model.to(DEVICE)

NeuralNetwork(
  (l1): Linear(in_features=98, out_features=32, bias=True)
  (act): ReLU()
  (outl): Linear(in_features=32, out_features=1, bias=True)
  (final_act): Sigmoid()
)

In [8]:
X_train, X_val, y_train , y_val = train_test_split(X_train, y_train, shuffle=True,
                                                    random_state=52, test_size=0.2)

In [9]:
X_train_pt = torch.tensor(X_train.to_numpy()).to(torch.float32).to(DEVICE)
X_test_pt = torch.tensor(X_test.to_numpy()).to(torch.float32).to(DEVICE)
X_val_pt = torch.tensor(X_val.to_numpy()).to(torch.float32).to(DEVICE)
y_train_pt = torch.tensor(y_train.to_numpy()).to(torch.float32).to(DEVICE)
y_test_pt = torch.tensor(y_test.to_numpy()).to(torch.float32).to(DEVICE)
y_val_pt = torch.tensor(y_val.to_numpy()).to(torch.float32).to(DEVICE)

(X_train_pt.shape, X_test_pt.shape, X_val_pt.shape,
 y_train_pt.shape, y_test_pt.shape, y_val_pt.shape)

(torch.Size([20838, 98]),
 torch.Size([6513, 98]),
 torch.Size([5210, 98]),
 torch.Size([20838, 1]),
 torch.Size([6513, 1]),
 torch.Size([5210, 1]))

In [10]:
class AdultDataSet(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __getitem__(self, index):
        return (self.X[index], self.y[index])

    def __len__(self):
        return len(self.X)

In [11]:
train_ds = AdultDataSet(X_train_pt, y_train_pt)
test_ds = AdultDataSet(X_test_pt, y_test_pt)
val_ds = AdultDataSet(X_val_pt, y_val_pt)

In [12]:
dl_train = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dl_test = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=True)
dl_val = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=True)

In [13]:
EPOCH_COUNT = 100

In [14]:
model = NeuralNetwork(98, 32).to(DEVICE)
loss_func = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [15]:
best_val_f1 = -1

for epoch in range(1, EPOCH_COUNT+1):
    model.train()

    for indx, (batch_x, batch_y) in enumerate(iter(dl_train), start=1):
        y_pred = model(batch_x) # [100, 1]
        loss = loss_func(y_pred, batch_y)  # порядок критичен!!!
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if indx % 50 == 0:
            y = batch_y.detach().cpu().numpy().astype(np.int64)
            y_pred = y_pred.detach().cpu().numpy()
            y_pred = np.where(y_pred > 0.5, 1, 0)
            f1_train = f1_score(y, y_pred, average='macro')
            print(f'[Train] F1: {f1_train:.4f}')

    model.eval()
    with torch.no_grad():
        f1s = []
        for batch_x, batch_y in dl_val:
            y_pred = model(batch_x)
            y = batch_y.cpu().numpy().astype(np.int64)
            y_pred = y_pred.cpu().numpy()
            y_pred = np.where(y_pred > 0.5, 1, 0)
            f1_val = f1_score(y, y_pred, average='macro')
            f1s.append(f1_val)
        f1_val_mean = torch.tensor(f1s).mean()

        if f1_val_mean > best_val_f1:
            best_val_f1 = f1_val_mean
            torch.save(model.state_dict(), 'models/best_model_weights.pth')

        print('-' * 100)
        print(f'[Val epoch={epoch:2d}] F1: {f1_val_mean:.4f}')
        print('-' * 100)

[Train] F1: 0.4919
[Train] F1: 0.5362
[Train] F1: 0.5464
[Train] F1: 0.6186
----------------------------------------------------------------------------------------------------
[Val epoch= 1] F1: 0.5828
----------------------------------------------------------------------------------------------------
[Train] F1: 0.6863
[Train] F1: 0.6678
[Train] F1: 0.6078
[Train] F1: 0.7890
----------------------------------------------------------------------------------------------------
[Val epoch= 2] F1: 0.6707
----------------------------------------------------------------------------------------------------
[Train] F1: 0.7137
[Train] F1: 0.7611
[Train] F1: 0.6889
[Train] F1: 0.6710
----------------------------------------------------------------------------------------------------
[Val epoch= 3] F1: 0.7211
----------------------------------------------------------------------------------------------------
[Train] F1: 0.7900
[Train] F1: 0.7086
[Train] F1: 0.7188
[Train] F1: 0.8125
------------

In [16]:
model.load_state_dict(
    torch.load('models/best_model_weights.pth', weights_only=True)
)

<All keys matched successfully>

In [17]:
# финальный тест

model.eval()

with torch.no_grad():
    ys = []
    preds = []
    for batch_x, batch_y in dl_test:
        y_pred = model(batch_x)
        y = batch_y.cpu().numpy().astype(np.int64)
        y_pred = y_pred.cpu().numpy()
        y_pred = np.where(y_pred > 0.5, 1, 0)
        ys.extend(y.tolist())
        preds.extend(y_pred.tolist())

    f1_test = f1_score(ys, preds, average='macro')

f1_test

0.7955181891008367

### Гипотеза 2
1й скрытый слой будет фича-экстрактором размерности 256, 2й слой - экстрактор фичей более высокого уровня, 3й - так же выходной.

In [142]:
class NeuralNetwork(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.l1 = nn.Linear(in_features, 64)
        self.l2 = nn.Linear(64, 1)
        self.act = nn.ReLU()
        self.final_act = nn.Sigmoid()

    def forward(self, x):
        y = self.l1(x)
        y = self.act(y)
        y = self.l2(y)
        y = self.final_act(y)
        return y

In [143]:
EPOCH_COUNT = 100

In [144]:
model = NeuralNetwork(98).to(DEVICE)
loss_func = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [145]:
best_val_f1 = -1
best_epoch = -1

for epoch in range(1, EPOCH_COUNT+1):
    model.train()

    for indx, (batch_x, batch_y) in enumerate(iter(dl_train), start=1):
        y_pred = model(batch_x) # [100, 1]
        loss = loss_func(y_pred, batch_y)  # порядок критичен!!!
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        f1s = []
        for batch_x, batch_y in dl_val:
            y_pred = model(batch_x)
            y = batch_y.cpu().numpy().astype(np.int64)
            y_pred = y_pred.cpu().numpy()
            y_pred = np.where(y_pred > 0.5, 1, 0)
            f1_val = f1_score(y, y_pred, average='macro')
            f1s.append(f1_val)
        f1_val_mean = torch.tensor(f1s).mean()

        if f1_val_mean > best_val_f1:
            best_val_f1 = f1_val_mean
            best_epoch = epoch
            torch.save(model.state_dict(), 'models/best_model_weights.pth')

        print(f'[Val epoch={epoch:2d}] F1: {f1_val_mean:.4f}')

print(f'Best epoch = {best_epoch}')

[Val epoch= 1] F1: 0.5744
[Val epoch= 2] F1: 0.6932
[Val epoch= 3] F1: 0.7265
[Val epoch= 4] F1: 0.7510
[Val epoch= 5] F1: 0.7541
[Val epoch= 6] F1: 0.7604
[Val epoch= 7] F1: 0.7593
[Val epoch= 8] F1: 0.7559
[Val epoch= 9] F1: 0.7594
[Val epoch=10] F1: 0.7629
[Val epoch=11] F1: 0.7693
[Val epoch=12] F1: 0.7642
[Val epoch=13] F1: 0.7640
[Val epoch=14] F1: 0.7738
[Val epoch=15] F1: 0.7678
[Val epoch=16] F1: 0.7674
[Val epoch=17] F1: 0.7750
[Val epoch=18] F1: 0.7751
[Val epoch=19] F1: 0.7796
[Val epoch=20] F1: 0.7776
[Val epoch=21] F1: 0.7770
[Val epoch=22] F1: 0.7829
[Val epoch=23] F1: 0.7803
[Val epoch=24] F1: 0.7839
[Val epoch=25] F1: 0.7837
[Val epoch=26] F1: 0.7792
[Val epoch=27] F1: 0.7735
[Val epoch=28] F1: 0.7805
[Val epoch=29] F1: 0.7781
[Val epoch=30] F1: 0.7783
[Val epoch=31] F1: 0.7810
[Val epoch=32] F1: 0.7840
[Val epoch=33] F1: 0.7831
[Val epoch=34] F1: 0.7788
[Val epoch=35] F1: 0.7845
[Val epoch=36] F1: 0.7814
[Val epoch=37] F1: 0.7754
[Val epoch=38] F1: 0.7765
[Val epoch=3

In [146]:
model.load_state_dict(
    torch.load('models/best_model_weights.pth', weights_only=True)
)

<All keys matched successfully>

In [147]:
# финальный тест

model.eval()

with torch.no_grad():
    ys = []
    preds = []
    for batch_x, batch_y in dl_test:
        y_pred = model(batch_x)
        y = batch_y.cpu().numpy().astype(np.int64)
        y_pred = y_pred.cpu().numpy()
        y_pred = np.where(y_pred > 0.5, 1, 0)
        ys.extend(y.tolist())
        preds.extend(y_pred.tolist())

    f1_test = f1_score(ys, preds, average='macro')

f1_test

0.798367968633906